# Dataset Validation

In this notebook, we will validate the raw dataset for the Human Message Rewriter project.

The goal is to make sure each dataset row has the correct format before we use it for fine-tuning.

Each row should contain:

- id
- category
- instruction
- input
- output

# Importing Libraries


In [1]:
import json
from pathlib import Path

# Inspect Dataset

In [2]:
dataset_path = Path("../data/raw/human_rewriter_raw.jsonl")
dataset_path


WindowsPath('../data/raw/human_rewriter_raw.jsonl')

In [3]:
dataset_path.exists()

True

In [5]:
with dataset_path.open("r",encoding="utf-8") as f:
    for i, line in enumerate(f):
        print(line.strip())

        if i == 2:
            break

{"id": "ex_0001", "category": "workplace", "instruction": "Rewrite this message so it sounds natural, clear, and human while keeping the same meaning.", "input": "Hello team, I am writing this message to inform you that I will be late by approximately 10 minutes.", "output": "Hey everyone, I’m running about 10 minutes late.", "source": "synthetic_curated"}
{"id": "ex_0002", "category": "workplace", "instruction": "Rewrite this message so it sounds natural, clear, and human while keeping the same meaning.", "input": "I wanted to inform you that I have completed the task that was assigned to me.", "output": "I’ve completed the task you assigned me.", "source": "synthetic_curated"}
{"id": "ex_0003", "category": "workplace", "instruction": "Rewrite this message so it sounds natural, clear, and human while keeping the same meaning.", "input": "Please be informed that I have attached the required document for your reference.", "output": "I’ve attached the required document for your reference

In [6]:
with dataset_path.open("r",encoding="utf-8") as file:
    first_line = file.readline()

first_row = json.loads(first_line)
first_row

{'id': 'ex_0001',
 'category': 'workplace',
 'instruction': 'Rewrite this message so it sounds natural, clear, and human while keeping the same meaning.',
 'input': 'Hello team, I am writing this message to inform you that I will be late by approximately 10 minutes.',
 'output': 'Hey everyone, I’m running about 10 minutes late.',
 'source': 'synthetic_curated'}

In [7]:
print("Input")
print(first_row["input"])

Input
Hello team, I am writing this message to inform you that I will be late by approximately 10 minutes.


In [8]:
print("Output")
print(first_row['output'])

Output
Hey everyone, I’m running about 10 minutes late.


## Required fields

In [27]:
REQUIRED_FIELDS = {
    "category",
    "input",
    "output",
    "instruction"
}
REQUIRED_FIELDS

{'category', 'input', 'instruction', 'output'}


## loading all rows into a list

In [12]:
rows = []
with dataset_path.open("r",encoding="utf-8") as file:
    for line_number, line in enumerate(file):
        line = line.strip()

        if not line:
            continue

        row = json.loads(line)
        rows.append(row)

len(rows)


                       

300

## Inspect one loaded row

In [14]:
rows[1]

{'id': 'ex_0002',
 'category': 'workplace',
 'instruction': 'Rewrite this message so it sounds natural, clear, and human while keeping the same meaning.',
 'input': 'I wanted to inform you that I have completed the task that was assigned to me.',
 'output': 'I’ve completed the task you assigned me.',
 'source': 'synthetic_curated'}

In [18]:
print("----Input----")
print(rows[0]['input'])
print("----Output----")
print(rows[0]['output'])

----Input----
Hello team, I am writing this message to inform you that I will be late by approximately 10 minutes.
----Output----
Hey everyone, I’m running about 10 minutes late.


## Check if any fields are missing in any row

In [28]:
missing_field_errors = []
for no, line in enumerate(rows):
    for field in REQUIRED_FIELDS:
        if field not in line:
            missing_field_errors.append(f"Missing field: {field} in line number: {no+1}")
missing_field_errors[:10]


[]

## look for empty fields

In [29]:
empty_field_errors = []

for no, line in enumerate(rows):
    for field in REQUIRED_FIELDS:
        if field in line and not line[field].strip():
            empty_field_errors.append(f"Empty field: {field} in line number: {no+1}")
empty_field_errors[:10]

[]

## Look for Duplicate ids

In [30]:
seen_ids = set()
duplicate_id_errors = []
no_id_errors = []

for no, line in enumerate(rows):
    if "id" in line:
        if line["id"] in seen_ids:
            duplicate_id_errors.append(f"Duplicate id: {line['id']} in line number: {no+1}")
        else:
            seen_ids.add(line["id"])
    else:
        no_id_errors.append(f"No id field in line number: {no+1}")
duplicate_id_errors[:10]
no_id_errors[:10]


[]



## Number of categories

In [33]:
category_counts= {}

for row in rows:
    category = row["category"]
    category_counts[category] = category_counts.get(category, 0)+1
category_counts

{'workplace': 35,
 'student_message': 35,
 'formal_email': 30,
 'follow_up': 30,
 'reminder': 30,
 'wordy_to_concise': 40,
 'polite_request': 30,
 'awkward_casual': 35,
 'clearer_text': 35}

In [34]:
for category, count in sorted(category_counts.items()):
    print(f"{category}: {count}")

awkward_casual: 35
clearer_text: 35
follow_up: 30
formal_email: 30
polite_request: 30
reminder: 30
student_message: 35
wordy_to_concise: 40
workplace: 35


# Summary

In [36]:
print("Dataset Validation Summary")
print("-" * 30)
print(f"Total rows: {len(rows)}")
print(f"Unique IDs: {len(seen_ids)}")
print(f"Number of categories: {len(category_counts)}")
print(f"Missing field errors: {len(missing_field_errors)}")
print(f"Empty field errors: {len(empty_field_errors)}")
print(f"Duplicate ID errors: {len(duplicate_id_errors)}")

Dataset Validation Summary
------------------------------
Total rows: 300
Unique IDs: 300
Number of categories: 9
Missing field errors: 0
Empty field errors: 0
Duplicate ID errors: 0


### Duplicate ID and Category Validation

In this section, we checked that:

- every example has a unique ID
- the dataset has multiple categories
- the category distribution is reasonably balanced

This matters because fine-tuning works better when the dataset is clean, traceable, and diverse.